In [1]:
import os
import math
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

Używane urządzenie: cuda


In [2]:
# =====================================================================
# 1. PRZYGOTOWANIE DANYCH I PARAMETRÓW BAZOWYCH
# =====================================================================

# Definicja 7 cech (w tym cech z AGGRESCAN-u wyliczonych wcześniej)
FEATURE_COLUMNS = [
    'beta_propensity',
    'proline_fraction',
    'net_charge',
    'polar_fraction',
    'AAT',
    'TA',
    'a3vSA'
]

path = "df_ml_good_with_features.csv"
df = pd.read_csv(path)
df = df[df["class"] == 1].copy()

MAX_LEN = 50
PAD_IDX = 0
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i+1: aa for i, aa in enumerate(AMINO_ACIDS)}

EOS = len(AMINO_ACIDS) + 1
VOCAB_SIZE = EOS + 1

# Skalowanie wszystkich 7 cech przed podziałem na zbiory
scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS])

def encode_sequence(seq):
    seq = seq[:MAX_LEN-1]
    tokens = [aa_to_idx.get(a, PAD_IDX) for a in seq]
    tokens.append(EOS)
    tokens += [PAD_IDX] * (MAX_LEN - len(tokens))
    return torch.tensor(tokens)

def decode_sequence(tokens):
    out = []
    for t in tokens:
        if t == EOS:
            break
        if t != PAD_IDX:
            out.append(idx_to_aa[t])
    return "".join(out)

class ProteinDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = encode_sequence(row["sequence"])
        features = torch.tensor(
            row[FEATURE_COLUMNS].astype(float).to_numpy(),
            dtype=torch.float32
        )
        return seq, features

# Podział danych i przygotowanie loaderów
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = ProteinDataset(train_df)
val_dataset = ProteinDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, drop_last=False)

In [ ]:
# =====================================================================
# 2. ARCHITEKTURA MODELU (GUIDED DIFFUSION TRANSFORMER)
# =====================================================================

class GuidedDiffusionTransformer(nn.Module):
    def __init__(self, vocab_size, feature_dim, differentiable_feat_dim, d_model=128, max_len=50, num_steps=100):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.num_steps = num_steps

        # embedding dla treningu
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Rzutowanie zaszumionego wektora x_t przed wejściem do Transformera
        self.input_projection = nn.Linear(d_model, d_model)

        # Blok kodowania czasu t
        self.time_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # Blok rzutowania cech wejściowych (Warunek)
        self.feature_proj = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, d_model)
        )

        # Rdzeń sieci
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=256, batch_first=True, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=4)

        # Głowa 1: Przewidywanie szumu
        self.noise_predictor = nn.Linear(d_model, d_model)

        # Głowa 2: Przewidywanie profilu cech (wspólny loss)
        self.feature_predictor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, differentiable_feat_dim)
        )

        self.to_logits = nn.Linear(d_model, vocab_size)

        # POPRAWKA: Definicja JEDNEGO spójnego harmonogramu liniowego dla treningu i generacji
        beta = torch.linspace(0.0001, 0.02, num_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)

        self.register_buffer("beta", beta)
        self.register_buffer("alpha", alpha)
        self.register_buffer("alpha_bar", alpha_bar)

    def get_time_embedding(self, timesteps):
        half_dim = self.d_model // 2
        emb = math.log(10000.0) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) * -emb)
        emb = timesteps.float().unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
        return self.time_mlp(emb)

    def forward(self, x_t, t, features):
        B, L, D = x_t.shape

        # Rzutowanie wejścia szumu, aby dopasować skalę
        x_t_proj = self.input_projection(x_t)

        t_emb = self.get_time_embedding(t).unsqueeze(1)  # [B, 1, D]
        f_emb = self.feature_proj(features).unsqueeze(1)  # [B, 1, D]
        cond_tokens = t_emb + f_emb                       # [B, 1, D]

        inp = torch.cat([cond_tokens, x_t_proj], dim=1)   # [B, 1 + L, D]
        h = self.transformer(inp)

        seq_h = h[:, 1:, :]

        pred_noise = self.noise_predictor(seq_h)
        latent_summary = seq_h.mean(dim=1)
        pred_differentiable_feats = self.feature_predictor(latent_summary)

        return pred_noise, pred_differentiable_feats

    def forward_diffusion(self, x_0, t):
        """Dodawanie szumu za pomocą ujednoliconego harmonogramu z bufora"""
        sqrt_alpha_bar = torch.sqrt(self.alpha_bar[t]).view(-1, 1, 1)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alpha_bar[t]).view(-1, 1, 1)

        epsilon = torch.randn_like(x_0)
        x_t = sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * epsilon
        return x_t, epsilon

In [ ]:
# =====================================================================
# 2. INICJALIZACJA I TRENING MODELU
# =====================================================================

# Przyjmujemy stałe z Twojego kodu: VOCAB_SIZE, FEATURE_COLUMNS, train_loader, val_loader
# pierwsze 4 cechy w FEATURE_COLUMNS są łatwo obliczalne matematycznie
DIFF_FEAT_DIM = 4
NUM_STEPS = 100

diff_model = GuidedDiffusionTransformer(
    vocab_size=VOCAB_SIZE,
    feature_dim=len(FEATURE_COLUMNS),
    differentiable_feat_dim=DIFF_FEAT_DIM,
    num_steps=NUM_STEPS
).to(device)

optimizer = torch.optim.AdamW(diff_model.parameters(), lr=5e-4, weight_decay=0.01)
diff_criterion = nn.MSELoss()
feat_criterion = nn.MSELoss()

best_val_loss = float("inf")
patience = 5
counter = 0
EPOCHS = 100
gamma = 0.5  # waga dla straty kontrolnej cech fizykochemicznych


In [ ]:
print("Rozpoczęcie treningu modelu dyfuzyjnego...")

for epoch in range(EPOCHS):
    # ----------------- TRENING -----------------
    diff_model.train()
    train_loss = 0

    for x, f in train_loader:
        x, f = x.to(device), f.to(device)
        f_diff_true = f[:, :DIFF_FEAT_DIM]  # Wycinamy cechy do dodatkowej straty

        optimizer.zero_grad()

        # Pobranie ciągłego embeddingu dla naturalnego białka
        with torch.no_grad():
            x_0 = diff_model.embedding(x)

        # Wylosowanie kroku czasowego t dla każdego przykładu w batchu
        t = torch.randint(0, diff_model.num_steps, (x.size(0),), device=device)

        # Proces dyfuzji w przód (zaszumianie)
        x_t, true_noise = diff_model.forward_diffusion(x_0, t)

        # Przejście przez sieć odszumiającą
        pred_noise, pred_feats = diff_model(x_t, t, f)

        # Obliczanie hybrydowej funkcji straty
        loss_diffusion = diff_criterion(pred_noise, true_noise)
        loss_features = feat_criterion(pred_feats, f_diff_true)
        loss_total = loss_diffusion + gamma * loss_features

        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(diff_model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss_total.item()

    train_loss /= len(train_loader)

    # ----------------- WALIDACJA -----------------
    diff_model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, f in val_loader:
            x, f = x.to(device), f.to(device)
            f_diff_true = f[:, :DIFF_FEAT_DIM]

            x_0 = diff_model.embedding(x)
            t = torch.randint(0, diff_model.num_steps, (x.size(0),), device=device)
            x_t, true_noise = diff_model.forward_diffusion(x_0, t)

            pred_noise, pred_feats = diff_model(x_t, t, f)

            loss_diffusion = diff_criterion(pred_noise, true_noise)
            loss_features = feat_criterion(pred_feats, f_diff_true)
            loss_total = loss_diffusion + gamma * loss_features

            val_loss += loss_total.item()

    val_loss /= len(val_loader)
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early Stopping oparty o zbalansowany loss walidacji
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        counter = 0
        torch.save(diff_model.state_dict(), "../../../../Downloads/best_diffusion.pt")
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping włączone.")
        break

# Załadowanie najlepszych wag po zakończeniu treningu
diff_model.load_state_dict(torch.load("../../../../Downloads/best_diffusion.pt"))
print("Model dyfuzyjny gotowy do generowania.")

In [6]:
diff_model.load_state_dict(torch.load("../../../../Downloads/best_diffusion.pt"))

<All keys matched successfully>

In [7]:
# =====================================================================
# 3. PROCEDURA GENEROWANIA SEKWENCJI (SAMPLING LOOP)
# =====================================================================

def generate_sequence_diffusion(model, features, max_len=50, num_steps=100, temperature=0.5):
    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        features = features.unsqueeze(0).to(device)  # [1, Feature_Dim]

        # 1. Inicjalizacja czystym szumem
        x_t = torch.randn((1, max_len, model.d_model)).to(device)

        # 2. Pętla odszumiająca według tego samego harmonogramu co trening
        for t_idx in reversed(range(num_steps)):
            t_tensor = torch.tensor([t_idx], device=device)

            pred_noise, _ = model(x_t, t_tensor, features)

            cur_beta = model.beta[t_idx]
            cur_alpha = model.alpha[t_idx]
            cur_alpha_bar = model.alpha_bar[t_idx]

            if t_idx > 0:
                noise = torch.randn_like(x_t)
                sigma = torch.sqrt((1.0 - model.alpha_bar[t_idx-1]) / (1.0 - cur_alpha_bar) * cur_beta)
            else:
                noise = 0
                sigma = 0

            # Równanie odszumiania DDPM
            x_t = (1.0 / torch.sqrt(cur_alpha)) * (
                x_t - (cur_beta / torch.sqrt(1.0 - cur_alpha_bar)) * pred_noise
            ) + sigma * noise

        # 3. Zamiana czystego embeddingu x_0 na Logity i tokeny
        logits = model.to_logits(x_t).squeeze(0)  # Zmiana kształtu na [max_len, VOCAB_SIZE]

        # POPRAWKA: Bezpieczna kontrola długości białka
        length_control_factor = 1.5
        for pos in range(15, max_len):
            logits[pos, EOS] += length_control_factor

        # Próbkowanie z temperaturą
        probs = torch.softmax(logits / temperature, dim=-1)
        tokens = torch.multinomial(probs, num_samples=1).squeeze(-1).cpu().numpy()

        return decode_sequence(tokens)

In [11]:
# =====================================================================
# 4. MASOWE GENEROWANIE I ZAPIS DO PLIKU CSV
# =====================================================================

generated_sequences = []
N_ROWS = 1006
N_PER_ROW = 20  # analogicznie jak w Twoich poprzednich skryptach

# Funkcja perturbacji cech identyczna z Twoją wersją dla cVAE
def perturb_features_diff(row, noise_level=0.1):
    feats = row[FEATURE_COLUMNS].values.astype(float)
    noise = np.random.normal(0, noise_level, size=len(feats))
    feats = feats + noise
    # Wykorzystujemy Twój obiekt scaler, który musi być dostępny w przestrzeni nazw
    feats = scaler.transform(feats.reshape(1, -1))[0]
    return torch.tensor(feats, dtype=torch.float32)

sample_df = df.sample(N_ROWS, random_state=42)
print("Rozpoczęcie generowania sekwencji syntetycznych...")

for _, row in sample_df.iterrows():
    for _ in range(N_PER_ROW):
        # Generowanie zaszumionego profilu cech fizykochemicznych
        feats = perturb_features_diff(row, noise_level=0.1)

        # Wywołanie dedykowanej pętli dyfuzyjnej
        seq = generate_sequence_diffusion(diff_model, feats, max_len=MAX_LEN, num_steps=NUM_STEPS)

        # Czyszczenie i sanityzacja stringu (Twoje reguły)
        seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
        if len(seq) >= 5:
            generated_sequences.append(seq)

# Zapis rezultatów końcowych do pliku
df_out = pd.DataFrame({
    "sequence": generated_sequences,
    "length": [len(s) for s in generated_sequences]
})

Rozpoczęcie generowania sekwencji syntetycznych...


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  

In [16]:
df_out.to_csv(f"Diff_generated_sequences-{len(generated_sequences)}.csv", index=False)
print(len(generated_sequences))

16334


In [15]:
for i in range(100):
  print(generated_sequences[i])

PNQDE
DCPVASIYEHSIE
NKQTVQRWTLPHVG
EYNEVEYDYTYW
LQDHALHEPATTIP
TYLNYGNARIDWTNF
VMEAPDMDP
YDDQSQVIAHRFH
LYHIIKWVARV
GTMSCFYVQGHKEQ
HHVVKYQHWN
QMWWGREFIYDTEH
TRNPEVYCVGCN
YLGRQY
SVDESFEEMKCGPY
EFFGLIVP
CELPVIYVIN
YHHERARPAKSNRGN
YNDSLI
ERDLYSLTWWPNK
HLTFWLPDDVWERYYHW
RQCNKLLVVPCPEDRYRNV
YLFPMVRVKQSHSSLM
FLPPDVFGNKICA
TGVTEEWNN
HKTNLCPPGTEDSY
GKSCVQDDRYTLAEG
VSHQGRWYREDCIQ
HDNRPEGCQFFIYD
HWHIEYNTSHLCGQDK
EVVILIFCGYHFWEVEGT
NKQFLYSGNEVYSLFT
VEQAEDLYKYHIDR
PMTKERRWWSMLGN
YYKVEVIIIGSSAQF
KKCNRKWRDHLNMHTLQ
EKHPHYNVCVDQ
WFPNVCKEYGKTCID
AFTHG
TQTDDALNVHYMS
MRTASRYTTSCKN
NYFSNNHLG
NEAPICTN
SLKETKQYLGDVGK
DHTSCK
LMTNQQMDIFEQYAQYN
RQHHGPGFTYNSQH
RWNRLKPWIHHKRQIR
KTYGISRDSHLTECS
CNHHVWPNMRVHHMYP
LNMNDGGGR
ENCVQWGVTDYNWNVSMY
GIPNRDKIPEHP
GDSEDDKTAQPVGFIGIN
TGSDIRIWHQGVCY
GAQVSCLADGWLETVLH
VMHRH
KDIVWDAVMPNMRQAY
DQVDGGPIPAIEDWN
PYLLYFGIHKDNWCT
DYWDEKDENTWHHTWGWVA
SEYEMNM
RDIWVI
CNKFGYQNTYNLNNRVDH
PSETTNQTSNVKGEC
QEDCSSIVVQYSTMNK
NIVDSSYHSRLCSV
HRELWGTILTQ
HDQLMKFECDVRLES
CAEGDKGPEQCHEGI
EKPPCPIHYEIIN